# Notebook 04: Pricing Validation

**Purpose**: validate the calibrated local-vol surface by repricing a sample of market options and comparing model prices to observed mid prices.

This notebook:
1. Loads the calibrated `LocalVolSurface` from Notebook 03
2. Loads cleaned market options from Notebook 01
3. Reprices a bounded sample of market calls with the PDE pricer
4. Produces repricing diagnostics by level, moneyness, and maturity


### What Pricing Validation Means Here
The local-vol surface is not useful just because it exists. It must be able to reproduce listed vanilla prices with tolerable error. This notebook therefore takes the calibrated local-vol surface, reprices a representative subset of market options with the PDE solver, and summarizes where the model is accurate and where it is weak.


In [ ]:
import sys
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")

from pricing.pde_solver import price_european_pde_local_vol, PDEConfig

OUTPUT_DIR = Path("../output")
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Imports complete")


In [ ]:
lv_surface_path = OUTPUT_DIR / "local_vol_surface.pkl"
market_path = OUTPUT_DIR / "cleaned_options_data.csv"

if not lv_surface_path.exists():
    raise FileNotFoundError(f"Missing {lv_surface_path}. Run Notebook 03 first.")
if not market_path.exists():
    raise FileNotFoundError(f"Missing {market_path}. Run Notebook 01 first.")

with open(lv_surface_path, "rb") as f:
    lv_surface = pickle.load(f)

df_market = pd.read_csv(market_path)
df_market = df_market[df_market["option_type"].astype(str).str.lower() == "call"].copy()

print(f"✅ Loaded LocalVolSurface with spot domain [{lv_surface.S_grid.min():.2f}, {lv_surface.S_grid.max():.2f}]")
print(f"✅ Loaded {len(df_market)} market call options")


The market sample used here is deliberately bounded. Full repricing of every contract is possible, but a representative subset is easier to inspect and already reveals whether errors are concentrated in short maturities, far wings, or elsewhere.


In [ ]:
df_market["moneyness"] = df_market["strike"] / df_market["underlying_price"]

mask = (
    df_market["mid"].notna()
    & (df_market["time_to_expiry"] > 0.0)
    & (df_market["time_to_expiry"] <= lv_surface.t_grid.max())
    & (df_market["strike"] >= lv_surface.S_grid.min())
    & (df_market["strike"] <= lv_surface.S_grid.max())
    & (df_market["underlying_price"] >= lv_surface.S_grid.min())
    & (df_market["underlying_price"] <= lv_surface.S_grid.max())
)

df_eval = df_market.loc[mask].copy()
if df_eval.empty:
    raise RuntimeError("No market options overlap the calibrated local-vol domain")

MAX_OPTIONS = 150
if len(df_eval) > MAX_OPTIONS:
    df_eval = df_eval.sample(MAX_OPTIONS, random_state=7).sort_values(["time_to_expiry", "strike"]).reset_index(drop=True)
else:
    df_eval = df_eval.sort_values(["time_to_expiry", "strike"]).reset_index(drop=True)

print(f"📊 Evaluating {len(df_eval)} options")
print(f"   T range: [{df_eval['time_to_expiry'].min():.4f}, {df_eval['time_to_expiry'].max():.4f}] years")
print(f"   strike range: [{df_eval['strike'].min():.2f}, {df_eval['strike'].max():.2f}]")


In [ ]:
pde_cfg = PDEConfig(n_space=220, n_time=120)
model_prices = []

for idx, row in df_eval.iterrows():
    if idx % 25 == 0:
        print(f"pricing {idx}/{len(df_eval)}")

    price = price_european_pde_local_vol(
        S0=float(row["underlying_price"]),
        K=float(row["strike"]),
        T=float(row["time_to_expiry"]),
        r=float(row["risk_free_rate"]),
        q=float(row["dividend_yield"]),
        option_type="call",
        lv_surface=lv_surface,
        cfg=pde_cfg,
    )
    model_prices.append(price)

df_eval["model_price_lv"] = model_prices
df_eval["pricing_error"] = df_eval["model_price_lv"] - df_eval["mid"]
df_eval["abs_pricing_error"] = df_eval["pricing_error"].abs()

print("✅ Repricing complete")
print(df_eval[["mid", "model_price_lv", "pricing_error"]].describe().to_string())


These plots connect the mathematical model back to observable market prices. If the local-vol surface is too rough, the repricing errors will usually show up first in the wings and at shorter maturities.


In [ ]:
fig = plt.figure(figsize=(14, 5))

ax1 = fig.add_subplot(1, 2, 1)
ax1.scatter(df_eval["mid"], df_eval["model_price_lv"], s=18, alpha=0.55)
mn = min(df_eval["mid"].min(), df_eval["model_price_lv"].min())
mx = max(df_eval["mid"].max(), df_eval["model_price_lv"].max())
ax1.plot([mn, mx], [mn, mx], linestyle="--", linewidth=1.2)
ax1.set_xlabel("Market Mid Price")
ax1.set_ylabel("LV PDE Price")
ax1.set_title("Local Vol Repricing: Model vs Market")
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(1, 2, 2)
ax2.hist(df_eval["pricing_error"], bins=35, alpha=0.85)
ax2.set_xlabel("LV Price - Market Mid")
ax2.set_ylabel("Count")
ax2.set_title("Repricing Error Distribution")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "05_repricing_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {FIG_DIR / '05_repricing_errors.png'}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_eval["moneyness"], df_eval["pricing_error"], s=18, alpha=0.55)
axes[0].axhline(0.0, linestyle="--", linewidth=1.0)
axes[0].set_xlabel("Moneyness K/S")
axes[0].set_ylabel("LV Price - Market Mid")
axes[0].set_title("Repricing Error vs Moneyness")
axes[0].grid(alpha=0.3)

axes[1].scatter(df_eval["time_to_expiry"], df_eval["pricing_error"], s=18, alpha=0.55)
axes[1].axhline(0.0, linestyle="--", linewidth=1.0)
axes[1].set_xlabel("Maturity T (years)")
axes[1].set_ylabel("LV Price - Market Mid")
axes[1].set_title("Repricing Error vs Maturity")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "05_repricing_by_moneyness.png", dpi=150, bbox_inches="tight")
plt.savefig(FIG_DIR / "05_repricing_by_maturity.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {FIG_DIR / '05_repricing_by_moneyness.png'}")
print(f"✅ Saved: {FIG_DIR / '05_repricing_by_maturity.png'}")


In [ ]:
df_eval.to_csv(OUTPUT_DIR / "repricing_validation.csv", index=False)

print("=" * 60)
print("NOTEBOOK 04 SUMMARY: PRICING VALIDATION")
print("=" * 60)
print(f"Options repriced:        {len(df_eval)}")
print(f"Mean error:              {df_eval['pricing_error'].mean():.6f}")
print(f"Mean absolute error:     {df_eval['abs_pricing_error'].mean():.6f}")
print(f"RMSE:                    {np.sqrt(np.mean(df_eval['pricing_error'] ** 2)):.6f}")
print("✅ Notebook 4 Complete: Pricing Validation")
